In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
pd.set_option("display.max_columns",None)
sns.set_style("whitegrid")

In [8]:
df = pd.read_csv("loan.csv")
df.shape


C:\Users\CK\AppData\Local\Temp\ipykernel_16976\2357451083.py:1: DtypeWarning: Columns (0: desc, 1: verification_status_joint) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("loan.csv")


(887379, 74)

In [9]:
selected_cols = [
    # Loan info
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    
    # Borrower financials
    "annual_inc",
    "dti",
    
    # Credit behavior
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "total_acc",
    "revol_bal",
    "revol_util",
    
    # Borrower profile
    "emp_length",
    "home_ownership",
    "verification_status",
    "purpose",
    
    # Dates (for feature engineering)
    "issue_d",
    "earliest_cr_line",
    
    # Target
    "loan_status"
]
df  = df[selected_cols]

In [10]:
df.columns


Index(['loan_amnt', 'term', 'int_rate', 'installment', 'annual_inc', 'dti',
       'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'total_acc', 'revol_bal',
       'revol_util', 'emp_length', 'home_ownership', 'verification_status',
       'purpose', 'issue_d', 'earliest_cr_line', 'loan_status'],
      dtype='str')

In [11]:
df.shape


(887379, 19)

In [12]:
df.isna().sum()

loan_amnt                  0
term                       0
int_rate                   0
installment                0
annual_inc                 4
dti                        0
delinq_2yrs               29
inq_last_6mths            29
open_acc                  29
total_acc                 29
revol_bal                  0
revol_util               502
emp_length             44825
home_ownership             0
verification_status        0
purpose                    0
issue_d                    0
earliest_cr_line          29
loan_status                0
dtype: int64

In [13]:
df["emp_length"] = df["emp_length"].fillna("Unknown")

In [14]:
num_cols = [
    "annual_inc",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "total_acc",
    "revol_util"
]

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [15]:
num_cols = [
    "annual_inc",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "total_acc",
    "revol_util"
]

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [16]:
df["issue_d"] = pd.to_datetime(df["issue_d"], errors="coerce")
df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], errors="coerce")

C:\Users\CK\AppData\Local\Temp\ipykernel_16976\43061412.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["issue_d"] = pd.to_datetime(df["issue_d"], errors="coerce")
C:\Users\CK\AppData\Local\Temp\ipykernel_16976\43061412.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], errors="coerce")


In [17]:
df["credit_age_days"] = (df["issue_d"] - df["earliest_cr_line"]).dt.days

In [18]:
df["credit_age_days"] = df["credit_age_days"].fillna(df["credit_age_days"].median())

In [19]:
df.drop(["issue_d", "earliest_cr_line"], axis=1, inplace=True)


In [20]:
df.isna().sum()

loan_amnt              0
term                   0
int_rate               0
installment            0
annual_inc             0
dti                    0
delinq_2yrs            0
inq_last_6mths         0
open_acc               0
total_acc              0
revol_bal              0
revol_util             0
emp_length             0
home_ownership         0
verification_status    0
purpose                0
loan_status            0
credit_age_days        0
dtype: int64

In [21]:
def map_status(status):
    if status == "Fully Paid":
        return 0   # Good borrower
    elif status in ["Charged Off", "Default"]:
        return 1   # Bad borrower
    else:
        return None  # Ignore unfinished loans

In [22]:
df["default"] = df["loan_status"].apply(map_status)

In [23]:
#Remove unclear outcomes
df = df[df["default"].notnull()]
#Convert to integer
df["default"] = df["default"].astype(int)
#drop the original oan status colum 
df.drop("loan_status", axis=1, inplace=True)

In [ ]:
# categoroical columns
df.select_dtypes(include="object").columns.tolist()

C:\Users\CK\AppData\Local\Temp\ipykernel_16976\3671663173.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include="object").columns.tolist()


['term', 'emp_length', 'home_ownership', 'verification_status', 'purpose']

In [31]:
df['term'].value_counts()

term
36 months    197373
60 months     56817
Name: count, dtype: int64

In [37]:
### cleaning the categorical data 

#1. term: "36 months" -> 36, "60 months" -> 60
df["term"] = df["term"].astype(str).str.extract(r'(\d+)').astype(int)


In [36]:
df['emp_length'].value_counts()

emp_length
10    77256
0     30943
2     23647
3     20484
5     18136
1     16951
4     16263
6     14816
7     14156
8     11922
9      9616
Name: count, dtype: int64

In [41]:

#2. emp_length: "10+ years" -> 10, "< 1 year" -> 0, "Unknown" -> 0

def clean_emp_length(val):
    if pd.isna(val):
        return 0
    
    val = str(val).strip()
    
    if val == "Unknown":
        return 0
    elif "<" in val:
        return 0
    elif "+" in val:
        return 10
    else:
        return int(val.split()[0])


#call the emp_length function 
df["emp_length"] = df["emp_length"].apply(clean_emp_length)




In [43]:
#Encode remaining categorical columns
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include='object').columns

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

C:\Users\CK\AppData\Local\Temp\ipykernel_16976\1303816663.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include='object').columns


In [46]:
df.dtypes

loan_amnt                int64
term                     int64
int_rate               float64
installment            float64
annual_inc             float64
dti                    float64
delinq_2yrs            float64
inq_last_6mths         float64
open_acc               float64
total_acc              float64
revol_bal                int64
revol_util             float64
emp_length               int64
home_ownership           int64
verification_status      int64
purpose                  int64
credit_age_days        float64
default                  int64
dtype: object

In [47]:
df.isnull().sum().sum()

np.int64(0)

In [49]:
df.shape
df.head()

,loan_amnt,term,int_rate,installment,annual_inc,dti,delinq_2yrs,inq_last_6mths,open_acc,total_acc,revol_bal,revol_util,emp_length,home_ownership,verification_status,purpose,credit_age_days,default
0,5000,36,10.65,162.87,24000.0,27.65,0.0,1.0,3.0,9.0,13648,83.7,10,5,2,1,-724297.0,0
1,2500,60,15.27,59.83,30000.0,1.00,0.0,5.0,3.0,4.0,1687,9.4,0,5,1,0,-729500.0,1
2,2400,36,15.96,84.33,12252.0,8.72,0.0,2.0,2.0,10.0,2956,98.5,10,5,0,11,40.0,0
3,10000,36,13.49,339.31,49200.0,20.00,0.0,1.0,10.0,37.0,5598,21.0,10,5,1,9,-728345.0,0
5,5000,36,7.90,156.46,36000.0,11.20,0.0,3.0,9.0,12.0,7963,28.3,3,5,1,13,37.0,0


In [51]:
df["emp_length"].value_counts().sort_index()

emp_length
0     30943
1     16951
2     23647
3     20484
4     16263
5     18136
6     14816
7     14156
8     11922
9      9616
10    77256
Name: count, dtype: int64

In [52]:
#MODEL A  TRAINING 
X = df.drop("default", axis=1)
y = df["default"]

In [53]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
df['x_train']